# 29 — Showcase on unquantized Llama 3.1 8B (bf16)

Notebook 26 concluded that Llama is the family outlier for multi-concept
steering (~50% both-improved vs 97–99% on Gemma). But that comparison is
confounded: the Gemma rows are bf16 while the Llama rows are **AWQ-INT4**.
This notebook runs the identical showcase design (30 pairs, 27 concepts,
single vs blend/vote/schedule, k=4, steps=24, 10 prompts) on
`meta-llama/Llama-3.1-8B-Instruct` in bf16 — separating model family from
quantization. If bf16 Llama jumps toward Gemma-like both-improved rates, the
outlier story becomes a quantization story; if it stays ~50%, the family
conclusion survives with the confound removed.

| model | source |
|---|---|
| `Meta-Llama-3.1-8B-Instruct-AWQ-INT4` | reused from notebook 24's log (as `llama-3.1-8b-awq`) |
| `meta-llama/Llama-3.1-8B-Instruct` | **run here**; bf16, sharded across GPUs (`device_map="auto"`) |
| `google/gemma-2-2b-it` | reused from notebook 25's log |
| `google/gemma-2-9b-it` | reused from notebook 26's log |

Notes specific to this comparison:

- The AWQ and bf16 variants share one tokenizer, so the WordNet concept
  data is identical; the **frames still differ** (the unembedding weights
  do), so ρ is recomputed and logged per model as usual.
- Same Llama chat template, `min_lemmas=11` (8 WordNet languages) — exactly
  the settings of every prior Llama multiconcept run.
- The AWQ vs bf16 contrast is within-family on identical pairs and prompts:
  it is the direct measure of what 4-bit quantization does to steering.


In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.evaluation.harness import EvaluationHarness
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept
from frames.representations.schedulers import RoundRobin
from frames.utils.memory import gc_cuda

In [2]:
MIN_LEMMAS = 11  # Llama concepts draw on 8 WordNet languages (multiconcept standard)
MAX_TOKENS = 3
K = 4
STEPS = 24

PAIRS = json.loads(Path("resources/15_e0_selected_pairs.json").read_text())["all_pairs"]
SYNSETS = sorted({p[side] for p in PAIRS for side in ("a", "b")})

MODELS = {
    "llama-3.1-8b-bf16": {
        "min_lemmas": MIN_LEMMAS,
        "kwargs": {
            "id": "meta-llama/Llama-3.1-8B-Instruct",
            "torch_dtype": torch.bfloat16,
            "device_map": "auto",
        },
    },
}
# reused from prior logs (identical design), not re-run:
LLAMA_AWQ_KEY = "llama-3.1-8b-awq"  # notebook 24 (logged there as llama-3.1-8b)
GEMMA2B_KEY = "gemma-2-2b"  # notebook 25
GEMMA9B_KEY = "gemma-2-9b"  # notebook 26
ALL_MODELS = [LLAMA_AWQ_KEY, *MODELS, GEMMA2B_KEY, GEMMA9B_KEY]


def chat_fn(model_key):
    if "llama" in model_key:
        return lambda u, a="": (
            f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>{a}"
        )
    if "gemma" in model_key:
        return lambda u, a="": (
            f"<start_of_turn>user\n{u}<end_of_turn>\n<start_of_turn>model\n{a}"
        )
    raise ValueError(model_key)


QUESTIONS = [
    ("What men can be?", "1."),
    ("What women can be?", "1."),
    ("Tell me a short story.", "Once"),
    ("Describe your favorite place.", "My favorite place is"),
    ("What do you like to do on weekends?", "I like"),
    ("Give me some advice for a good life.", "1."),
    ("What is the most important thing in the world?", "The most important thing is"),
    ("Describe the weather today.", "Today the weather is"),
    ("What should I cook for dinner?", "You should"),
    ("Name things that make people happy.", "1."),
]

LOG_PATH = Path("resources/29_llama8b_bf16.jsonl")
LOG_PATH.unlink(missing_ok=True)
print(f"{len(PAIRS)} pairs, {len(SYNSETS)} concepts, {len(MODELS)} models")

30 pairs, 27 concepts, 1 models


In [3]:
def run_model(model_key: str, spec: dict) -> None:
    min_lemmas = spec["min_lemmas"]
    fur = FrameUnembeddingRepresentation.from_model_id(
        **{"device_map": "cuda:0", **spec["kwargs"]}
    )
    chat = chat_fn(model_key)
    prompts = [chat(u, a) for u, a in QUESTIONS]

    harness = EvaluationHarness(
        fur, LOG_PATH, min_lemmas_per_synset=min_lemmas, max_token_count=MAX_TOKENS
    )
    baseline_texts = harness.generate_baseline(prompts, max_new_tokens=STEPS + 2)

    # tokenizer-dependent concept survival
    concept, missing = {}, []
    for name in SYNSETS:
        try:
            concept[name] = fur.get_concept_cached(name, min_lemmas, MAX_TOKENS)
        except Exception as error:  # noqa: BLE001 - survival is data-dependent
            missing.append(name)
            print(f"  DROPPED {name}: {type(error).__name__}: {error}")
    pairs = [p for p in PAIRS if p["a"] not in missing and p["b"] not in missing]
    print(f"[{model_key}] {len(concept)}/{len(SYNSETS)} concepts, {len(pairs)} pairs")
    if not pairs:
        # distinguishes systemic failure (e.g. OOM dropping every synset)
        # from ordinary data-dependent survival
        raise RuntimeError(f"[{model_key}] no pairs survived concept building")

    for name, c in concept.items():
        torch.cuda.empty_cache()
        texts, probe = fur.generate_with_topk_guide(prompts, guide=c, k=K, steps=STEPS)
        harness.evaluate(
            prompts,
            texts,
            [name],
            config={"model": model_key, "method": "single", "concept": name},
            probe=probe,
            baseline_texts=baseline_texts,
            concepts={name: c},
        )
    print(f"[{model_key}] singles done")

    for pair in pairs:
        a, b = pair["a"], pair["b"]
        ca, cb = concept[a], concept[b]
        rho_model = float(ca.rho(cb))
        methods = {
            "blend": lambda: fur.generate_with_topk_guide(
                prompts, guide=Concept.average([ca, cb]), k=K, steps=STEPS
            ),
            "vote": lambda: fur.generate_with_topk_multi_guide(
                prompts, guides=[ca, cb], k=K, steps=STEPS, normalize="zscore"
            ),
            "schedule": lambda: fur.generate_with_topk_multi_guide(
                prompts, guides=[ca, cb], k=K, steps=STEPS, scorer=RoundRobin()
            ),
        }
        for method, generate in methods.items():
            torch.cuda.empty_cache()
            texts, probe = generate()
            harness.evaluate(
                prompts,
                texts,
                [a, b],
                config={
                    "model": model_key,
                    "method": method,
                    "a": a,
                    "b": b,
                    "rho": rho_model,
                    "rho_llama": pair["rho"],
                    "regime": pair["regime"],
                },
                probe=probe,
                baseline_texts=baseline_texts,
                concepts={a: ca, b: cb},
            )
        print(f"[{model_key}] {a} + {b} (rho={rho_model:.2f}): done")


for model_key, spec in MODELS.items():
    with gc_cuda():
        run_model(model_key, spec)

Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


/home/rodrigo/projects/frh-multi/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:801: UserWarning: Not enough free disk space to download the file. The expected file size is: 4976.70 MB. The target location cache/hub/models--meta-llama--Llama-3.1-8B-Instruct/blobs only has 2386.74 MB free disk space.
  warnings.warn(


model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

RuntimeError: Failed to load model: Data processing error: CAS service error : IO Error: No space left on device (os error 28)

In [ ]:
parsed = [json.loads(line) for line in LOG_PATH.read_text().strip().split("\n")]
models_seen = {r["config"]["model"] for r in parsed}
assert models_seen == set(MODELS), models_seen

# AWQ-Llama rows: notebook 24's records (identical design), mapped onto the
# descriptive method names and re-keyed to distinguish the quantized variant.
METHOD_MAP = {
    "single": "single",
    "F1.a-mean": "blend",
    "F2.a-wsum": "vote",
    "F3.a-roundrobin": "schedule",
}
for r in [
    json.loads(line)
    for line in Path("resources/24_showcase.jsonl").read_text().strip().split("\n")
]:
    r["config"] = {
        **r["config"],
        "model": LLAMA_AWQ_KEY,
        "method": METHOD_MAP[r["config"]["method"]],
    }
    parsed.append(r)

# Gemma rows: notebooks 25 and 26, reused verbatim (same design and names).
for path, key in (
    ("resources/25_multimodel.jsonl", GEMMA2B_KEY),
    ("resources/26_gemma9b.jsonl", GEMMA9B_KEY),
):
    parsed.extend(
        r
        for r in [
            json.loads(line) for line in Path(path).read_text().strip().split("\n")
        ]
        if r["config"]["model"] == key
    )

assert all(r["expression"] is not None for r in parsed)
print(
    f"gate: {len(parsed)} records across {sorted({r['config']['model'] for r in parsed})}"
)

rows = []
for r in parsed:
    c = r["config"]
    if c["method"] == "single":
        rows.append(
            {
                "model": c["model"],
                "method": "single",
                "delta_a": r["expression"][c["concept"]]["delta"],
                "delta_b": None,
                "joint": None,
                "rho": None,
                "regime": None,
                "ppl_ratio": r["ppl_ratio"],
            }
        )
    else:
        da = r["expression"][c["a"]]["delta"]
        db = r["expression"][c["b"]]["delta"]
        rows.append(
            {
                "model": c["model"],
                "method": c["method"],
                "delta_a": da,
                "delta_b": db,
                "joint": min(da, db),
                "rho": c["rho"],
                "regime": c["regime"],
                "ppl_ratio": r["ppl_ratio"],
            }
        )
results = pd.DataFrame(rows)

summary = (
    results.groupby(["model", "method"])
    .agg(
        delta_a=("delta_a", "mean"),
        delta_b=("delta_b", "mean"),
        joint=("joint", "mean"),
        both_positive=("joint", lambda s: (s.dropna() > 0).mean()),
        ppl_ratio=("ppl_ratio", "mean"),
        n=("ppl_ratio", "size"),
    )
    .reset_index()
    .sort_values(["model", "joint"], ascending=[True, False])
)
summary.to_csv("resources/29_multimodel_summary.csv", index=False)
summary.round(3)

In [ ]:
multi = summary[summary["method"] != "single"]
order = ["blend", "vote", "schedule"]
colors = {"blend": "tab:blue", "vote": "tab:orange", "schedule": "tab:green"}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, (metric, label) in zip(
    axes,
    (
        ("joint", "mean joint expression gain (per-model scale)"),
        ("both_positive", "share of runs with BOTH concepts improved"),
    ),
):
    for j, method in enumerate(order):
        vals = [
            multi[(multi["model"] == m) & (multi["method"] == method)][metric].iloc[0]
            for m in ALL_MODELS
        ]
        ax.bar(
            [i + (j - 1) * 0.25 for i in range(len(ALL_MODELS))],
            vals,
            width=0.24,
            color=colors[method],
            label=method,
        )
    ax.set_xticks(range(len(ALL_MODELS)), ALL_MODELS, fontsize=8)
    ax.set_ylabel(label)
    ax.axhline(0, color="gray", lw=0.8)
axes[0].legend(fontsize=8)
axes[1].axhline(0.5, color="gray", ls="--", lw=0.8)
plt.suptitle("Multi-concept steering: quantization vs model family (30 pairs)")
plt.tight_layout()
plt.savefig("resources/29_llama_dequant.png", dpi=150)
plt.show()

print("within-model method ranking by joint:")
for m in ALL_MODELS:
    ranked = multi[multi["model"] == m].sort_values("joint", ascending=False)
    print(f"  {m}: " + " > ".join(ranked["method"]))

print("\nAWQ -> bf16 (within Llama, identical pairs):")
for method in order:
    awq = multi[(multi["model"] == LLAMA_AWQ_KEY) & (multi["method"] == method)]
    bf16 = multi[(multi["model"] == "llama-3.1-8b-bf16") & (multi["method"] == method)]
    print(
        f"  {method:9s} joint {awq['joint'].iloc[0]:+.3f} -> "
        f"{bf16['joint'].iloc[0]:+.3f}   both-improved "
        f"{awq['both_positive'].iloc[0]:.0%} -> {bf16['both_positive'].iloc[0]:.0%}"
    )

In [ ]:
def answer(text: str, model_key: str) -> str:
    for marker in ("assistant<|end_header_id|>", "<start_of_turn>model\n"):
        if marker in text:
            return text.split(marker)[-1].strip()
    return text


for m in ALL_MODELS:
    for r in parsed:
        c = r["config"]
        if (
            c["model"] == m
            and c["method"] == "blend"
            and (c.get("a"), c.get("b")) == ("sadness.n.01", "music.n.01")
            and "short story" in r["prompt"]
        ):
            deltas = {
                k.split(".")[0]: round(v["delta"], 2)
                for k, v in r["expression"].items()
            }
            print(f"[{m} / blend] {deltas}")
            print("   ", answer(r["text"], m)[:150], "\n")
            break